In [15]:
import torch
import torch.nn as nn
import timm
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
import os
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

class CFG:
    BASE_PATH = 'csiro-biomass'
    SEED = 42
    N_FOLDS = 5
    MODEL_DIR = './tree_models'
    MODEL_NAME = "catboost_dino_v1" # Identifier
    TRAIN_CSV = os.path.join(BASE_PATH, 'train.csv')
    TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, 'train')
    BATCH_SIZE   = 1
    IMG_SIZE = 518

    TARGET_COLS    = ['Dry_Total_g', 'GDM_g', 'Dry_Green_g']
    DERIVED_COLS   = ['Dry_Clover_g', 'Dry_Dead_g']
    ALL_TARGET_COLS = ['Dry_Green_g','Dry_Dead_g','Dry_Clover_g','GDM_g','Dry_Total_g']
    R2_WEIGHTS     = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # matches metric

def get_extractor_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE), # CFG.IMG_SIZE must be 336
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std =[0.229, 0.224, 0.225]),
        ToTensorV2()
    ], p=1.0)
# ==========================================
# 1. DEFINE THE EXTRACTOR
# ==========================================
class ViTBiomassExtractor(nn.Module):
    def __init__(self, model_name='vit_small_patch14_dinov2.lvd142m'):
        super().__init__()
        # Load DINOv2. num_classes=0 means "Give me the feature vector, not the prediction"
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feat_dim = self.backbone.num_features
        print(f"Loaded {model_name}. Output Feature Dim: {self.feat_dim} per image.")

    def forward(self, left, right):
        # Extract features for Left and Right images independently
        f_left = self.backbone(left)
        f_right = self.backbone(right)
        
        # Concatenate them into one giant vector (e.g., 384 + 384 = 768)
        return torch.cat([f_left, f_right], dim=1)

# ==========================================
# 2. EXTRACTION FUNCTION
# ==========================================
def extract_dataset(model, loader, device):
    model.eval()
    
    # Storage
    feats_list = []
    meta_list = []   # Species, State
    cont_list = [] 
    target_list = [] # The 5 targets
    group_list = []  # Session IDs for splitting
    
    print("Starting Extraction...")
    with torch.no_grad():
        for l, r, lab, aux_cat, aux_cont in tqdm(loader):
            l, r = l.to(device), r.to(device)
            
            # 1. Run ViT
            # Output shape: [Batch, Feature_Dim * 2]
            features = model(l, r)
            
            # 2. Store Data (Move to CPU/Numpy immediately to save VRAM)
            feats_list.append(features.cpu().numpy().astype(np.float32))
            
            # Store Metadata (Cat indices: Species, State)
            meta_list.append(aux_cat.numpy())

            cont_list.append(aux_cont.numpy())
            
            # Store Targets (All 5 columns usually in 'lab')
            target_list.append(lab.numpy())
            
    # Stack into giant arrays
    X_img = np.vstack(feats_list)
    X_cat = np.vstack(meta_list)
    X_cont = np.vstack(cont_list)
    y_all = np.vstack(target_list)
    
    return X_img, X_cat, X_cont, y_all

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df        = df
        self.img_dir   = img_dir
        self.transform = transform
        self.paths     = df['image_path'].values
        self.labels    = df[CFG.ALL_TARGET_COLS].values.astype(np.float32)
        self.aux_cat_labels = df[CFG.CAT_AUX_COLS].values.astype(np.int64)
        self.aux_cont_labels = df[CFG.CONT_AUX_COLS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(self.img_dir, os.path.basename(self.paths[idx]))
        img  = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w, _ = img.shape
        mid = w // 2
        left  = img[:, :mid]
        right = img[:, mid:]
        left  = self.transform(image=left)['image']
        right = self.transform(image=right)['image']

        label = torch.from_numpy(self.labels[idx])
        aux_cat_label   = torch.from_numpy(self.aux_cat_labels[idx])
        aux_cont_label  = torch.from_numpy(self.aux_cont_labels[idx])
        return left, right, label, aux_cat_label, aux_cont_label

# ==========================================
# 3. EXECUTE
# ==========================================
# Assuming 'full_loader' is a DataLoader containing ALL your training data
# with NO random augmentations (Resize + Normalize only).


print("Loading data...")
df_long = pd.read_csv(CFG.TRAIN_CSV)
df_wide = df_long.pivot(index='image_path', columns='target_name', values='target').reset_index()
df_wide = df_wide[['image_path'] + CFG.ALL_TARGET_COLS]
print(f"{len(df_wide)} training images")

# Aux task
aux_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']
df_aux = df_long[aux_cols].drop_duplicates().reset_index(drop=True)

df_wide = df_wide.merge(df_aux, on='image_path', how='left')

df_wide['State_idx'],   STATE_MAP   = pd.factorize(df_wide['State'])
df_wide['Species_idx'], SPECIES_MAP = pd.factorize(df_wide['Species'])

CFG.N_STATES   = len(STATE_MAP)
CFG.N_SPECIES  = len(SPECIES_MAP)
print(f"Found {CFG.N_STATES} states and {CFG.N_SPECIES} species.")

# 2. Convert Date to cyclical features (we'll predict these)
df_wide['Sampling_Date'] = pd.to_datetime(df_wide['Sampling_Date'])
df_wide['day_of_year'] = df_wide['Sampling_Date'].dt.dayofyear
df_wide['day_sin'] = np.sin(2 * np.pi * df_wide['day_of_year'] / 365.25)
df_wide['day_cos'] = np.cos(2 * np.pi * df_wide['day_of_year'] / 365.25)
CFG.CONT_AUX_COLS = ['Pre_GSHH_NDVI', 'Height_Ave_cm', 'day_sin', 'day_cos']
CFG.CAT_AUX_COLS  = ['State_idx', 'Species_idx']
device = 'cuda'
extractor = ViTBiomassExtractor().to(device)

dataset= BiomassDataset(df_wide, CFG.TRAIN_IMAGE_DIR, transform=get_extractor_transforms())

loader  = DataLoader(dataset,  batch_size=CFG.BATCH_SIZE, shuffle=False)

# Run extraction
X_features, X_cat, X_cont, y_targets = extract_dataset(extractor, loader, device)

# Save to disk (so you don't have to run this again)
np.save('vit_features.npy', X_features)
np.save('vit_cat_meta.npy', X_cat)
np.save('vit_cont_meta.npy', X_cont)
np.save('vit_targets.npy', y_targets)

print(f"Saved Features shape: {X_features.shape}")
print(f"Saved Targets shape: {y_targets.shape}")

Loading data...
357 training images
Found 4 states and 15 species.
Loaded vit_small_patch14_dinov2.lvd142m. Output Feature Dim: 384 per image.
Starting Extraction...


100%|██████████| 357/357 [00:43<00:00,  8.26it/s]

Saved Features shape: (357, 768)
Saved Targets shape: (357, 5)


In [9]:
from datetime import datetime

from sklearn.model_selection import StratifiedGroupKFold


group_name = f"Date_{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"

config = {k: v for k, v in vars(CFG).items() if not k.startswith('_')}
df_wide['group'] = df_wide['State'].astype(str) + "_" + df_wide['Sampling_Date'].astype(str)
df_wide['biomass_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=10, labels=False)

# 4. GENERATE FOLDS
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=694)

# Create a new column 'fold' and fill it with -1
df_wide['fold'] = -1

print("Assigning folds...")
split_generator = sgkf.split(
    X=df_wide, 
    y=df_wide['biomass_bin'], 
    groups=df_wide['group']
)

for fold_idx, (train_idx, val_idx) in enumerate(split_generator):
    df_wide.loc[val_idx, 'fold'] = fold_idx
    print(f"Fold {fold_idx}: {len(val_idx)} validation samples.")

# 5. SORT AND SAVE
# Critical: Sort by image_path to align with your Feature Extraction order
df_wide = df_wide.sort_values('image_path').reset_index(drop=True)

save_path = 'train_folds_strict.csv'
df_wide.to_csv(save_path, index=False)

print(f"\nSUCCESS! Saved fold definitions to {save_path}")
print(f"Total Rows: {len(df_wide)}")

Assigning folds...
Fold 0: 72 validation samples.
Fold 1: 76 validation samples.
Fold 2: 71 validation samples.
Fold 3: 71 validation samples.
Fold 4: 67 validation samples.

SUCCESS! Saved fold definitions to train_folds_strict.csv
Total Rows: 357


In [18]:
# ====================================================
# TREE TRAINING NOTEBOOK (CATBOOST)
# Structure matches NN Pipeline for direct comparison
# ====================================================

import numpy as np
import pandas as pd
import os
import gc
import csv
from datetime import datetime
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor, Pool
import matplotlib.pyplot as plt

# --- CONFIG (Matches your NN CFG) ---

    
os.makedirs(CFG.MODEL_DIR, exist_ok=True)

# ====================================================
# 1. METRIC FUNCTION (Exact Match to PyTorch)
# ====================================================
def global_weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    """
    Computes the globally weighted R² score.
    y_true, y_pred: shape (N, 5)
    weights: Derived from CFG.R2_WEIGHTS
    """
    # Create a weight matrix of shape (N, 5)
    weights_matrix = np.tile(CFG.R2_WEIGHTS, (y_true.shape[0], 1))
    
    # 1. Calculate Weighted Mean of Truth (y_bar_w)
    weighted_sum = np.sum(weights_matrix * y_true)
    total_weight = np.sum(weights_matrix)
    y_bar_w = weighted_sum / total_weight 
    
    # 2. Calculate Residual Sum of Squares (SS_res)
    ss_res = np.sum(weights_matrix * (y_true - y_pred) ** 2)
    
    # 3. Calculate Total Sum of Squares (SS_tot)
    ss_tot = np.sum(weights_matrix * (y_true - y_bar_w) ** 2)
    
    # 4. Final Score
    # Safety to avoid division by zero if SS_tot is extremely small
    if ss_tot < 1e-7:
        return 0.0
        
    r2_w = 1 - (ss_res / ss_tot)
    return r2_w

# ====================================================
# 2. LOAD & PREPARE DATA (Matches NN Setup)
# ====================================================
print("Loading extracted features...")
# Load the files you created in the Extraction Step
X_img = np.load('vit_features.npy')    # DINOv2 Features
X_cat = np.load('vit_cat_meta.npy')    # [Species, State] indices
X_cont = np.load('vit_cont_meta.npy')
y_all = np.load('vit_targets.npy')     # [Green, Dead, Clover, GDM, Total]

# Load DataFrame to get Groups (Same as NN)
df = pd.read_csv('train_folds_strict.csv') # Ensure this matches the NN DF order!
groups = df['group'].values
bins = df['biomass_bin'].values

# Construct Training DataFrame
feat_cols = [f'dim_{i}' for i in range(X_img.shape[1])]
df_X = pd.DataFrame(X_img, columns=feat_cols)

# Add Metadata (CatBoost handles these natively)
# df_X['Species'] = X_cat[:, 0].astype(int)
# df_X['State']   = X_cat[:, 1].astype(int)

# Optional: Add Continuous (Height/NDVI) IF PREDICTED (Student Mode)
# If these are Ground Truth, DO NOT include them for Kaggle comparison.
df_X['NDVI']   = X_cont[:, 0]
df_X['Height'] = X_cont[:, 1]

# Targets: We train on the 3 Independent Roots
# Indices based on your NN: Green(0), GDM(3), Total(4)
y_roots = y_all[:, [0, 3, 4]] 
df_y = pd.DataFrame(y_roots, columns=['Dry_Green_g', 'GDM_g', 'Dry_Total_g'])

print(f"Data Loaded: {df_X.shape}")
print(f"Groups Found: {len(np.unique(groups))}")

# ====================================================
# 3. MAIN 5-FOLD LOOP
# ====================================================
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

oof_preds_5col = np.zeros((len(df), 5))
cat_features = ['Species', 'State']
all_fold_best_scores = []

print(f"\n{'Fold':<5} | {'Val Weighted R2':<15} | {'Green R2':<10} | {'Total R2':<10}")
print("-" * 55)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(df_X, bins, groups=groups)):
    
    # 1. Split Data
    X_train, y_train = df_X.iloc[train_idx], df_y.iloc[train_idx]
    X_val, y_val     = df_X.iloc[val_idx],   df_y.iloc[val_idx]
    
    # 2. Define Model (MultiRMSE = Optimized MSE for 3 targets)
    model = CatBoostRegressor(
        iterations=5000,
        learning_rate=0.03, # Lower is better for features
        depth=6,
        loss_function='MultiRMSE', 
        eval_metric='MultiRMSE',
        task_type="GPU",
        devices='0',
        random_seed=CFG.SEED,
        verbose=0, # Silent training
        early_stopping_rounds=200,
        allow_writing_files=False,
        boosting_type='Plain'
    )
    
    # 3. Train
    model.fit(
        X_train, y_train,
        cat_features=None,
        eval_set=(X_val, y_val),
        use_best_model=True
    )
    
    # 4. Predict Roots [Green, GDM, Total]
    p_roots = model.predict(X_val)
    
    p_green = p_roots[:, 0]
    p_gdm   = p_roots[:, 1]
    p_total = p_roots[:, 2]
    
    # 5. POST-PROCESSING (Replicating NN 'Derived' Logic)
    # Enforce Physics
    p_gdm   = np.maximum(p_gdm, 0) # No negative mass
    p_total = np.maximum(p_total, p_gdm) # Total >= GDM
    p_green = np.maximum(p_green, 0)
    p_green = np.minimum(p_green, p_gdm) # Green <= GDM
    
    # Derive Missing Columns
    p_clover = np.clip(p_gdm - p_green, 0, None)
    p_dead   = np.clip(p_total - p_gdm, 0, None)
    
    # Stack [Green, Dead, Clover, GDM, Total]
    p_full = np.stack([p_green, p_dead, p_clover, p_gdm, p_total], axis=1)
    oof_preds_5col[val_idx] = p_full
    
    # 6. Evaluate (Using exact Ground Truth for Val)
    y_val_gt = y_all[val_idx] 
    fold_score = global_weighted_r2_score(y_val_gt, p_full)
    all_fold_best_scores.append(fold_score)
    
    # Save Model
    model.save_model(os.path.join(CFG.MODEL_DIR, f'catboost_fold{fold}.cbm'))
    
    # Logging
    r2_green = r2_score(y_val_gt[:, 0], p_full[:, 0])
    r2_total = r2_score(y_val_gt[:, 4], p_full[:, 4])
    print(f"{fold+1:<5} | {fold_score:<15.4f} | {r2_green:<10.4f} | {r2_total:<10.4f}")

# ====================================================
# 4. SUMMARY & LOGGING
# ====================================================
mean_cv_score = np.mean(all_fold_best_scores)
std_cv_score  = np.std(all_fold_best_scores)

# Calculate Global OOF Score using your function on the full arrays
global_oof_score = global_weighted_r2_score(y_all, oof_preds_5col)

print('\n' + '='*70)
print('         FINAL CROSS-VALIDATION SCORE')
print('='*70)
print(f'Public LB Score: 0.58') # Placeholder/Target
print(f'Local CV Score: {mean_cv_score:.3f} ± {std_cv_score:.3f}')
print(f'Global OOF Score: {global_oof_score:.4f}') # Added this as it's useful
print('\nIndividual Fold Scores:')
for i, score in enumerate(all_fold_best_scores):
    print(f'  Fold {i+1}: {score:.4f}')

# Logging to CSV (Same format as NN)
log_file = os.path.join(CFG.MODEL_DIR, 'experiment_log_trees.csv')
log_data = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'cv_mean': f"{mean_cv_score:.5f}",
    'model': CFG.MODEL_NAME,
    'note': "DINOv2 Features + CatBoost (MultiRMSE)"
}

try:
    with open(log_file, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(log_data.keys()))
        if not os.path.isfile(log_file): writer.writeheader()
        writer.writerow(log_data)
    print(f"\nExperiment saved to {log_file}")
except Exception as e:
    print(f"Log Error: {e}")

Loading extracted features...
Data Loaded: (357, 770)
Groups Found: 30

Fold  | Val Weighted R2 | Green R2   | Total R2  
-------------------------------------------------------
1     | 0.6913          | 0.5999     | 0.5983    
2     | 0.5402          | 0.6186     | 0.3580    
3     | 0.7397          | 0.6487     | 0.6126    
4     | 0.7106          | 0.5456     | 0.6182    
5     | 0.7068          | 0.7694     | 0.5782    

         FINAL CROSS-VALIDATION SCORE
Public LB Score: 0.58
Local CV Score: 0.678 ± 0.071
Global OOF Score: 0.7153

Individual Fold Scores:
  Fold 1: 0.6913
  Fold 2: 0.5402
  Fold 3: 0.7397
  Fold 4: 0.7106
  Fold 5: 0.7068

Experiment saved to ./tree_models/experiment_log_trees.csv
